# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [ ]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents


SyntaxError: invalid syntax (2365248899.py, line 9)

In [ ]:
# Initialize and constants
from openai import OpenAI
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


"Here's one:\n\nDid you know that honey never spoils? Archaeologists have found pots of honey in ancient Egyptian tombs that are over 3,000 years old and still perfectly edible! This is because honey is the only food that bees produce that humans cannot spoil or ferment. Its unique combination of acidity and water content makes it airtight and resistant to bacterial growth, allowing it to remain stable for thousands of years.\n\nWould you like another fun fact?"

In [5]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [11]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [12]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [13]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [17]:
def select_relevant_links(url):
    response = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {
                "role": "system",
                "content": link_system_prompt + "\n\nRespond ONLY with valid JSON."
            },
            {
                "role": "user",
                "content": get_links_user_prompt(url)
            }
        ],
        temperature=0  # helps keep output deterministic«
    )

    result = response.choices[0].message.content.strip()

    try:
        return json.loads(result)
    except json.JSONDecodeError:
        print("Raw output:", result)
        raise

In [18]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'careers/jobs page',
   'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [22]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling ollama")
    response = ollama.chat.completions.create(
         model="llama3.2",
        messages=[
            {
                "role": "system",
                "content": link_system_prompt + "\n\nRespond ONLY with valid JSON."
            },
            {
                "role": "user",
                "content": get_links_user_prompt(url)
            }
        ],
        temperature=0  # helps keep output deterministic«
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [23]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling ollama
Found 3 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'careers/jobs page',
   'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [24]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling ollama
Found 7 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co'},
  {'type': 'company page', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog', 'url': 'https://discuss.huggingface.co'},
  {'type': 'github', 'url': 'https://github.com/huggingface'},
  {'type': 'twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'linkedin company page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'join/discord', 'url': 'https://join.discord.huggingface.co'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [31]:
from urllib.parse import urlparse, urljoin

def is_valid_link(base_url, link):
    if not link:
        return False

    link = link.strip()

    # ❌ Skip junk
    if link.startswith("#") or link.startswith("javascript") or link.startswith("mailto"):
        return False

    # Convert relative → absolute
    full_url = urljoin(base_url, link)

    parsed = urlparse(full_url)

    # ❌ Only allow http/https
    if parsed.scheme not in ("http", "https"):
        return False

    # ✅ Optional: restrict to same domain
    base_domain = urlparse(base_url).netloc
    if parsed.netloc != base_domain:
        return False

    return full_url


def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)

    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    print(result)

    seen = set()

    for link in relevant_links.get('links', []):
        raw_url = link.get("url")

        full_url = is_valid_link(url, raw_url)
        if not full_url:
            print("Skipping invalid:", raw_url)
            continue

        # 🔁 Deduplicate
        normalized = full_url.rstrip("/")
        if normalized in seen:
            continue
        seen.add(normalized)

        result += f"\n\n### Link: {link.get('type', 'Unknown')}\n"

        try:
            result += fetch_website_contents(full_url)
        except Exception as e:
            print("Failed to fetch:", full_url, e)

    return result

In [32]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling ollama
Found 7 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
zai-org/GLM-5.1
Updated
1 day ago
•
35.9k
•
1.1k
openbmb/VoxCPM2
Updated
5 days ago
•
9.3k
•
776
google/gemma-4-31B-it
Updated
3 days ago
•
2.44M
•
1.79k
dealignai/Gemma-4-31B-JANG_4M-CRACK
Updated
3 days ago
•
107k
•
966
MiniMaxAI/MiniMax-M2.7
Updated
about 2 hours ago
•
18.3k
•
563
Browse 2M+ models
Spaces
Running
on
Zero
Featured
405
OmniVoice
🌍
405
High-quality voice cloning TTS for 600+ languages
Running
on
Zero
MCP
2.01k
Wan2.2 14B Preview
🐌
2.01k
generate a video from an image with a text prompt
Running
on
A10G
Featured
295
VoxCP

In [41]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [29]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [33]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling ollama
Found 7 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
zai-org/GLM-5.1
Updated
1 day ago
•
35.9k
•
1.1k
openbmb/VoxCPM2
Updated
5 days ago
•
9.3k
•
776
google/gemma-4-31B-it
Updated
3 days ago
•
2.44M
•
1.79k
dealignai/Gemma-4-31B-JANG_4M-CRACK
Updated
3 days ago
•
107k
•
966
MiniMaxAI/MiniMax-M2.7
Updated
about 2 hours ago
•
18.3k
•
563
Browse 2M+ models
Spaces
Running
on
Zero
Featured
405
OmniVoice
🌍
405
High-quality voice cloning TTS for 600+ languages
Running
on
Zero
MCP
2.01k
Wan2.2 14B Preview
🐌
2.01k
generate a video from an image with a text prompt
Running
on
A10G
Featured
295
VoxCP

'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nzai-org/GLM-5.1\nUpdated\n1 day ago\n•\n35.9k\n•\n1.1k\nopenbmb/VoxCPM2\nUpdated\n5 days ago\n•\n9.3k\n•\n776\ngoogle/gemma-4-31B-it\nUpdated\n3 days ago\n•\n2.44M\n•\n1.79k\ndealignai/Gemma-4-31B-JANG_4M-CRACK\nUpdated\n3 days ago\n•\n107k\n•\n966\nMiniMaxAI/MiniMax-M2.7\nUpdated\nabout 2 hours ago\n•\n18.3k\n•\n563\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n405\nOmniVoice

In [35]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [38]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling ollama
Found 7 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
zai-org/GLM-5.1
Updated
1 day ago
•
35.9k
•
1.1k
openbmb/VoxCPM2
Updated
5 days ago
•
9.3k
•
776
google/gemma-4-31B-it
Updated
3 days ago
•
2.44M
•
1.79k
dealignai/Gemma-4-31B-JANG_4M-CRACK
Updated
3 days ago
•
107k
•
966
MiniMaxAI/MiniMax-M2.7
Updated
about 2 hours ago
•
18.3k
•
564
Browse 2M+ models
Spaces
Running
on
Zero
Featured
405
OmniVoice
🌍
405
High-quality voice cloning TTS for 600+ languages
Running
on
Zero
MCP
2.01k
Wan2.2 14B Preview
🐌
2.01k
generate a video from an image with a text prompt
Running
on
A10G
Featured
295
VoxCP

# About Us
## Mission & Culture
At Hugging Face, we are dedicated to building a collaborative platform for the machine learning community. Our mission is to empower anyone to learn, contribute, and share knowledge in the field of AI. We strive to create an inclusive environment where people from diverse backgrounds can come together, share their ideas, and work towards a more open and ethical AI future.

Our company culture values innovation, transparency, and inclusivity. We believe that by working together, we can achieve great things and make a meaningful impact on the world. Our team is passionate about machine learning and committed to making it accessible to everyone.

## Meet our Team
Our team consists of talented individuals who share a common passion for machine learning and collaboration. From experienced engineers to researchers and educators, our team members come from diverse backgrounds and expertise, ensuring that we can tackle complex problems and find innovative solutions.

## Customers
We serve the most ambitious organizations, startups, and individuals in the field of AI. Our platform has enabled numerous projects, applications, and innovations that have transformed various industries, including healthcare, finance, education, and more. We are committed to providing exceptional support and service to our customers, ensuring they can focus on creating groundbreaking work.

## Careers
Join us at Hugging Face if you want to be part of a vibrant community of machine learning professionals, researchers, and enthusiasts. Our careers page features job openings in various categories, including software development, engineering, product management, marketing, sales, and more.

We offer competitive salaries, flexible working hours, remote work options, comprehensive benefits packages, and opportunities for professional growth and development. If you're passionate about machine learning, collaboration, and innovation, let's connect and explore how we can work together to build a better future.

## Enterprise & Partnerships
For organizations looking to accelerate their AI initiatives, we offer tailored solutions that cater to your unique needs. Our expert team provides consulting services on AI strategy, model development, data integration, deployment, and maintenance.

We partner with leading companies in the AI space to develop cutting-edge technologies and showcase its capabilities. Let's discuss how we can work together to achieve common goals and create innovative applications that make a meaningful impact.

## About Us
Hugging Face is more than just a platform – it's a community. Our home for HuggingFace is where you'll find the most used open-source models, datasets and applications.
Our vision is a collaboration platform that empowers everyone to build machines that think like humans.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [39]:
def stream_brochure(company_name, url):
    stream = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [40]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling ollama
Found 7 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
zai-org/GLM-5.1
Updated
1 day ago
•
35.9k
•
1.1k
openbmb/VoxCPM2
Updated
5 days ago
•
9.3k
•
777
google/gemma-4-31B-it
Updated
3 days ago
•
2.44M
•
1.79k
dealignai/Gemma-4-31B-JANG_4M-CRACK
Updated
3 days ago
•
107k
•
967
MiniMaxAI/MiniMax-M2.7
Updated
about 2 hours ago
•
18.3k
•
564
Browse 2M+ models
Spaces
Running
on
Zero
Featured
405
OmniVoice
🌍
405
High-quality voice cloning TTS for 600+ languages
Running
on
Zero
MCP
2.01k
Wan2.2 14B Preview
🐌
2.01k
generate a video from an image with a text prompt
Running
on
A10G
Featured
295
VoxCP

# Hugging Face Brochure

## Introduction

Hugging Face is a revolutionary platform that empowers the machine learning community to collaborate, experiment, and innovate with open-source models, datasets, and applications. Founded on the vision of building an ethical AI future, Hugging Face has become the hub for the collaboration and sharing of knowledge among ML engineers, scientists, and end-users.

## Mission

Our mission is to create a platform that facilitates collaboration, creativity, and learning across the machine learning community. We strive to make cutting-edge technology accessible to everyone, promoting an open and inclusive approach to AI development.

## Features and Benefits

*   A vast library of over 2 million pre-trained models and datasets, enabling users to experiment with various techniques and applications.
*   An intuitive collaboration platform that allows individuals to share, discover, and build upon each other's work.
*   Open-source stack that accelerates innovation and reduces barriers to entry for new projects.
*   A range of tools and platforms for exploring multiple modalities, including text, image, video, audio, and even 3D.

## Values

We believe in the importance of:

*   Diversity: Fostering an inclusive environment where people from all backgrounds can contribute and learn from each other.
*   Ethics: Prioritizing transparency, fairness, and accountability in AI development and application.
*   Creativity: Encouraging experimentation, innovation, and risk-taking to push the boundaries of what's possible with machine learning.

## Community Engagement

Our vibrant community is made up of passionate individuals who share our vision of building a better future through collaboration and sharing. Join us today to become part of this dynamic group!

## Careers & Opportunities

Looking for a role that aligns your passions with cutting-edge AI technology? Explore our [careers page](link) to discover job openings, learn about company culture, and meet the talented team behind Hugging Face.

[Call-to-action]

Join us in shaping the future of machine learning and creating an open and accessible world for all.

In [42]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling ollama
Found 7 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
zai-org/GLM-5.1
Updated
1 day ago
•
35.9k
•
1.1k
openbmb/VoxCPM2
Updated
5 days ago
•
9.3k
•
777
google/gemma-4-31B-it
Updated
3 days ago
•
2.44M
•
1.79k
dealignai/Gemma-4-31B-JANG_4M-CRACK
Updated
3 days ago
•
107k
•
967
MiniMaxAI/MiniMax-M2.7
Updated
about 2 hours ago
•
18.3k
•
564
Browse 2M+ models
Spaces
Running
on
Zero
Featured
405
OmniVoice
🌍
405
High-quality voice cloning TTS for 600+ languages
Running
on
Zero
MCP
2.01k
Wan2.2 14B Preview
🐌
2.01k
generate a video from an image with a text prompt
Running
on
A10G
Featured
295
VoxCP

**Welcome to Hugging Face: Where Collaboration Meets Innovation**

Imagine a world where artificial intelligence is built on collaboration, transparency, and creativity. At Hugging Face, we're making that vision a reality. Our platform is designed for machine learning enthusiasts, researchers, and practitioners to come together, share knowledge, and push the boundaries of what's possible.

**Our Mission**

We believe that AI should be a force for good, and that's why we've built a platform that empowers anyone to participate in the development of open-source machine learning models. Our goal is to create an inclusive community that fosters creativity, diversity, and innovation in the field of AI.

**What We Do**

Our collaboration platform hosts unlimited public models, datasets, and applications, making it easy for anyone to contribute or build upon existing work. We offer a range of tools and resources to help you explore, experiment, and learn from the vast array of machine learning models available.

**Meet Our Community**

Hugging Face is home to a vibrant community of machine learning enthusiasts, researchers, and developers from all over the world. From experts in natural language processing to computer vision researchers, our community is united by a passion for building intelligent systems that can make a positive impact on society.

**Join Our Team**

If you're passionate about AI, innovation, and collaboration, we want to hear from you. Our team is growing rapidly, and we're always looking for talented individuals to join us on this exciting journey. From software engineers to product managers, researchers to sales teams, there's a place for you at Hugging Face.

**Accelerating ML Research**

At Hugging Face, we believe that AI should be accessible to everyone, regardless of their background or expertise. That's why we offer paid Compute and Enterprise solutions, providing our customers with the resources they need to build and deploy machine learning models at scale.

**Explore Our Features**

Our platform offers a range of features that make it easy for you to explore, collaborate on, and learn from machine learning models. From text to image recognition, and video generation to natural language processing, our library has something for everyone.

**Say Hello to Hugging Face!**

We're not just building a platform – we're building a community that's shaping the future of AI. Join us today and be part of a movement that's making machine learning more accessible, transparent, and creative.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>